In [1]:
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"
os.environ["OMP_NUM_THREADS"] = "1"
import torch
torch.set_num_threads(1)
from dotenv import load_dotenv
import pandas as pd
import numpy as np
import time
import yaml
from google import genai
import sys
import importlib
import re
if 'faiss_index' in sys.modules:
    importlib.reload(sys.modules['faiss_index'])

from faiss_index import FAISSIndex


/Users/hoangntmbee/Downloads/job-crawler/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Data Prep

### Read file

In [2]:
# read csv 
df = pd.read_csv('assets/datasets/job_postings_samples_v2.csv')

# set seed, shuffle
df = df.sample(frac=1, random_state=42).reset_index(drop=True)

# remove duplications of title + description + location in df and only remain the first occurrence of job_posting_id
df = df.drop_duplicates(subset=['title', 'description', 'location']).reset_index(drop=True)

# remove rows with null description
df = df.dropna(subset=['description']).reset_index(drop=True)

# remove rows with empty description
df = df[df['description'].str.strip() != ''].reset_index(drop=True)

df.iloc[15]['description']

'BeaconFire is based in Central NJ, specializing in Software Development, Web Development, and Business Intelligence; looking for candidates with a strong background in Software Engineering or Computer Science for a C++ / Software Developer position Job Responsibilities: ● Develop, test, and maintain applications using C++ (C++11/14/17 and above) on the Linux platform. ● Write efficient, reliable, and maintainable code with a focus on performance and stability. ● Design and implement object-oriented and modular C++ components. ● Work with multithreading and concurrency mechanisms to build scalable and responsive systems. ● Participate in debugging, profiling, and performance optimization on Linux-based systems. ● Create, deploy, and maintain automated unit and system tests. ● Collaborate with testers to analyze reported defects and resolve issues in a timely manner. ● Support continuous improvement by researching alternative technologies and contributing to architectural and design dis

# Decoder

Free tier limits for gemma-3-27b-it:
- 15000 tokens/min
- 30 requests/min
- 14.4k requests/day

### Prompt test

In [3]:
# Ground truth labeling prompt test — multilabel mode (1 API call for all categories)

import json as _json

with open('assets/prompts.yaml', 'r') as f:
    prompts_config = yaml.safe_load(f)

ml_prompts    = prompts_config['models']['gemma-3-27b-it']['tasks']['ground_truth_labeling_multilabel']['prompts']
prompt_template   = ml_prompts['template']
temperature       = ml_prompts['temperature']
max_output_tokens = ml_prompts['max_output_tokens']
categories        = ml_prompts['categories']

# ── Sample job ────────────────────────────────────────────────────────────────
title = 'Senior Strategy Manager'

description = '''About Us: Advanced Systems Group, LLC enables creativity through better technology and operations for media creatives and content owners. From acquisition to delivery, on-premises or in the cloud, ASG ensures our clients’ success through tailored solutions. One of North America’s largest Media and Entertainment Technology and Operations suppliers, we provide engineering services, physical and cloud integration, training, support, and managed services. Our Managed Services deliver customized operations and services for all phases of media production, including creative and engineering. Founded in 1997, and providing nationwide services, ASG has teams based in North America, South America, and Europe.### We are looking for: Advanced Systems Group LLC (ASG) is seeking a Senior Strategy Manager to join our embedded Business Operations team at a media client in either Los Angeles or New York. This team is responsible for guiding content selection plans, including guiding capital & resource allocation decisions across the portfolio. The Business Operations team works closely with senior leaders across creative development, talent management, finance, legal, marketing and production, and has a high degree of visibility within the organization. The Senior Strategy role will be an essential member of the team, with broad analytical and strategic responsibilities to maximize portfolio performance, and deliver beloved content for our customers. This role is open to candidates in either Los Angeles, CA, or New York, NY, and is on-site at the client's offices 5 days/week. Responsibilities: Formulate top-down content strategy recommendations, including how we should size and allocate our content budget across 1P, 3P, and FAST content. Identify deep, actionable insights into the content preferences, behavior and needs of our customers, globally Perform extensive competitive analysis and benchmarking, and identify and report on market trends Forecast content performance, track performance and financial progress, and inform go-forward strategies to improve our outlook Author business-driving documents that surface key learnings and provide recommendations for our go-forward content offering Required Qualifications & Experience: 4 year degree in a relevant field 6+ years of experience in investment banking, management consulting, and/or strategy/business development at a major media or technology company Experience using complex modeling and analysis to inform key business decisions. Candidates must be highly proficient in excel, tableau, and other analytical tools. Preferred Qualifications & Experience: Excellent communication skills (both written and oral) Excellent analytical capabilities Strong work ethic and ability to work effectively with a high degree of autonomy Knowledge and passion for the entertainment industry, in particular subscription video on demand content Compensation & Benefits:This full-time role offers a salary of $170,000-200,000 depending on experience. At Advanced Systems Group, we prioritize an inclusive work environment and offer a variety of benefits to support our diverse team, including: Comprehensive medical coverage with 3 different plans to fit your needs, and 100% of the employee medical premium covered by ASG. Discounts on health and wellness programs, plus savings on travel and more. Voluntary benefits including disability, accident, critical illness insurance, and pet insurance. Employee Assistance Program offering counseling, financial coaching, and more. Paid time off to relax and recharge. Additional benefits to help you plan for the future, like life insurance and 401k. Interested applicants, including those from Washington state, may contact recruiting@asgllc.com to request a full disclosure of the benefits offerings. Advanced Systems Group LLC provides equal employment opportunities to all employees and applicants for employment and prohibits discrimination and harassment of any type without regard to race, color, religion, age, sex, national origin, disability status, genetics, protected veteran status, sexual orientation, gender identity or expression, or any other characteristic protected by federal, state or local laws.*
'''


load_dotenv('.env.dev.llm')
client = genai.Client(api_key=os.getenv('GEMINI_API_KEY'))

prompt = prompt_template.format(title=title, description=description[:1500])

print(f"Title: {title}")
print(f"Description: {description[:80]}...\n")
print(f"Categories to classify: {list(categories.keys())}\n")

response = client.models.generate_content(
    model="gemma-3-27b-it",
    contents=prompt,
    config=genai.types.GenerateContentConfig(
        temperature=temperature,
        max_output_tokens=max_output_tokens,
    )
)

raw = response.text.strip()
# gemma-3-27b-it does not support JSON mode; extract the JSON object from free text
match = re.search(r'\\{[^{}]*\\}', raw, re.DOTALL)
parsed = _json.loads(match.group()) if match else {}

# Enforce internship / entry_level mutual exclusion
if parsed.get("internship") == 1:
    parsed["entry_level"] = 0

print("Raw response:", raw)
print("\nParsed labels:")
for cat, label in parsed.items():
    flag = "✓" if label == 1 else "✗"
    print(f"  {flag} {cat}: {label}  — {categories.get(cat, '')}")


Title: Senior Strategy Manager
Description: About Us: Advanced Systems Group, LLC enables creativity through better technolo...

Categories to classify: ['internship', 'entry_level', 'data_analytics', 'software_engineering', 'finance_accounting', 'marketing', 'strategy_product_consulting']

Raw response: ```json
{"internship":0,"entry_level":0,"data_analytics":0,"software_engineering":0,"finance_accounting":0,"marketing":0,"strategy_product_consulting":1}
```

Parsed labels:


In [ ]:
# inspect check point in ground_truth

test = pd.read_csv('assets/ground_truths/ds_v2/llm_generated/checkpoint_labels.csv')


test = test.merge(df, on='job_posting_id', how='left')

# only keep internship	entry_level	data_analytics	software_engineering	finance_accounting	marketing, title, description

test = test[['title', 'description','internship', 'entry_level', 'data_analytics', 'software_engineering', 'finance_accounting', 'marketing', 'strategy_product_consulting']]

test

,title,description,internship,entry_level,data_analytics,software_engineering,finance_accounting,marketing,strategy_product_consulting
0,Paid Summer Marketing Intern,Continental Properties is looking for a Summer...,1,0,0,0,0,1,0
1,Embedded Engineer,Make your mark at Comcast -- a Fortune 30 glob...,0,0,0,1,0,0,0
2,Social Media & Email Marketing Manager,Job Description: We are seeking a Social Media...,0,0,0,0,0,1,0
3,"Senior Data Scientist, Innovation Lab - Remote","Job Posting - Salary Range: $153,075 - $275,53...",0,0,1,0,0,0,0
4,CRE Senior Analyst - Originations & Special Si...,Job Description: Our client is seeking a Senio...,0,0,0,0,1,0,0
...,...,...,...,...,...,...,...,...,...
992,MR Technical Trainer,Join us in pioneering breakthroughs in healthc...,0,0,0,0,0,0,0
993,"Recruiter, YouTube",This role is not eligible for U.S. immigration...,0,0,0,0,0,0,0
994,Electrical Engineer,Job Description Insight Global is seeking an S...,0,0,0,0,0,0,0
995,Stormwater Engineer,What Your Day Will Look Like: : As a stormwate...,0,0,0,0,0,0,0


### Generate summary column

In [5]:
# Load prompts configuration from YAML file
with open('assets/prompts.yaml', 'r') as f:
    prompts_config = yaml.safe_load(f)

# Extract gemma-3-27b-it model configuration
model_config = prompts_config['models']['gemma-3-27b-it']
summarization_config = model_config['tasks']['summarization']

# Get prompt templates (use v1 by default, can switch to v2)
prompt_main = summarization_config['prompts']['v4']

# Extract configuration values for use in next cells
prompt_template = prompt_main['template']
temperature = prompt_main['temperature']
max_output_tokens = prompt_main['max_output_tokens']


# Configuration for incremental processing
start_idx = 700
end_idx = 1000 # tmr start idx from 1874 
SUMMARY_FILE = 'assets/features/ds_v1/summaries_gemma-3-27b-it_v2.pkl'

# Load existing summaries if available, otherwise use base df
try:
    df_master = pd.read_pickle(SUMMARY_FILE)
    print(f"✓ Loaded existing summaries from '{SUMMARY_FILE}'")
    print(f"  Existing summaries: {df_master['summary'].notna().sum()}/{len(df_master)}")
except FileNotFoundError:
    df_master = df.copy()
    df_master['summary'] = None
    print(f"✓ No existing summary file found. Starting fresh.")

# Extract the slice to process
df1 = df_master.iloc[start_idx:end_idx].copy() # exclusive of end_idx

# Load Gemini API key
load_dotenv('.env.dev.llm')

# Initialize Gemini client
client = genai.Client(api_key=os.getenv("GEMINI_API_KEY"))

# Batch processing parameters
BATCH_SIZE = 10
RATE_LIMIT_DELAY =15  # seconds between batches to avoid exceeding 15k tokens/min

# Summarize descriptions using gemma-3-27b-it
print(f"\nSummarizing rows {start_idx} to {end_idx-1} ({len(df1)} job descriptions) using gemma-3-27b-it...")
print(f"Processing in {(len(df1) + BATCH_SIZE - 1) // BATCH_SIZE} batches of {BATCH_SIZE}")

summaries_list = []
total_batches = (len(df1) + BATCH_SIZE - 1) // BATCH_SIZE

for batch_num in range(total_batches):
    batch_start_idx = batch_num * BATCH_SIZE
    batch_end_idx = min(batch_start_idx + BATCH_SIZE, len(df1))
    batch_df = df1.iloc[batch_start_idx:batch_end_idx]
    
    print(f"\nSummarizing batch {batch_num + 1}/{total_batches} (jobs {batch_start_idx + 1}-{batch_end_idx})...")
    
    # Summarize each description in the batch
    for idx, row in batch_df.iterrows():
        job_description = str(row['description']) if pd.notna(row['description']) else ""

        summary = None
        max_retries = 2
        
        for attempt in range(max_retries + 1):
            try:
                # Generate summary using gemma
                response = client.models.generate_content(
                    model="gemma-3-27b-it",  # it: instruction-tuned
                    contents=prompt_template.replace("{description}", job_description).replace("{title}", str(row['title'])),
                    config=genai.types.GenerateContentConfig(
                        max_output_tokens=max_output_tokens,
                        temperature=temperature
                    )
                )
                summary = response.text.strip()
                break  # Success, exit retry loop
            except Exception as e:
                if attempt < max_retries:
                    print(f"  ✗ Error summarizing job {idx} (attempt {attempt + 1}/{max_retries + 1}): {e}")
                    print(f"  Retrying...")
                else:
                    print(f"  ✗ Failed after {max_retries + 1} attempts for job {idx}: {e}")
        
        summaries_list.append(summary)
    
    print(f"  ✓ Successfully summarized {len(batch_df)} descriptions")
    
    # Rate limiting: wait before next batch (except for the last batch)
    if batch_num < total_batches - 1:
        print(f"  Waiting {RATE_LIMIT_DELAY} seconds before next batch...")
        time.sleep(RATE_LIMIT_DELAY)

# Update master DataFrame with new summaries for this range
df_master.loc[start_idx:end_idx-1, 'summary'] = summaries_list

# Save updated master DataFrame
df_master.to_pickle(SUMMARY_FILE)
print(f"\n✓ Saved updated summaries to '{SUMMARY_FILE}'")
print(f"  Total summaries now: {df_master['summary'].notna().sum()}/{len(df_master)}")

print(f"\n✓ Completed summarization for {len(summaries_list)} job descriptions")
print("\n" + "="*60)
print("Sample Summaries:")
print("="*60)
for i in range(min(3, len(df1))):
    print(f"\nJob {i+1}: {df1.iloc[i]['title']}")
    print(f"Original length: {len(str(df1.iloc[i]['description']))} chars")
    print(f"Summary length: {len(summaries_list[i])} chars")
    print(f"Summary: {summaries_list[i][:200]}...")


✓ Loaded existing summaries from 'assets/features/ds_v1/summaries_gemma-3-27b-it_v2.pkl'
  Existing summaries: 1000/6883

Summarizing rows 700 to 999 (300 job descriptions) using gemma-3-27b-it...
Processing in 30 batches of 10

Summarizing batch 1/30 (jobs 1-10)...
  ✓ Successfully summarized 10 descriptions
  Waiting 15 seconds before next batch...

Summarizing batch 2/30 (jobs 11-20)...


KeyboardInterrupt: 

---


Software Engineer II (Oracle EBS Developer) position requires a Public Trust clearance, which will be applied for upon acceptance. The role involves coordinating and assisting with deployment and support activities, maintaining systems configuration, administering the Oracle E-Business Suite (EBS) application tier, patching with utilities like adpatch and ADOP, cloning EBS environments, and assisting with custom reports. It also includes participation in design meetings, support for testing, and collaboration with agile teams to deliver cloud-based solutions. 

Required skills and technologies include Oracle DBA experience (19c pluggable database) with software installation, patching, upgrades, tablespace management, account


# Embedding

In [ ]:
df = pd.read_pickle('assets/features/ds_v1/summaries_gemma-3-27b-it_v2.pkl')

df

Free tier limits for gemini-embedding-001:
- 30000 tokens/min
- 100 requests/min -> 1 job post = 1 request,  no exception to batch call
- 1000 requests/day

In [ ]:
import time

job_postings_batch = df[300:1000]

# Load Gemini API key
load_dotenv('.env.dev.llm')

# Initialize Gemini client
client = genai.Client(api_key=os.getenv("GEMINI_API_KEY"))

# Define combinations
combination1 = ['title']
combination2 = ['title', 'description']
combination3 = ['title', 'description', 'location']
combination4 = ['location']
combination5 = ['title', 'summary'] # summary column from decoder

# Batch processing parameters
BATCH_SIZE = 100
RATE_LIMIT_DELAY = 61  # seconds between batches to avoid exceeding 30k tokens/min

# Process in batches
embeddings_list = []
total_batches = (len(job_postings_batch) + BATCH_SIZE - 1) // BATCH_SIZE

print(f"Processing {len(job_postings_batch)} job postings in {total_batches} batches of {BATCH_SIZE}")

for batch_num in range(total_batches):
    start_idx = batch_num * BATCH_SIZE
    end_idx = min(start_idx + BATCH_SIZE, len(job_postings_batch))
    batch_df = job_postings_batch.iloc[start_idx:end_idx]
    
    print(f"\nProcessing batch {batch_num + 1}/{total_batches} (jobs {start_idx + 1}-{end_idx})...")
    
    # Prepare texts for this batch
    batch_texts = []
    for idx, row in batch_df.iterrows():
        # Convert to string and handle NaN values
        job_text = " ".join([str(row[col]) if pd.notna(row[col]) else "" for col in combination5])
        batch_texts.append(job_text.strip())
    
    # Generate embeddings for this batch
    try:
        result = client.models.embed_content(
            model="gemini-embedding-001",
            contents=batch_texts,
            # dimension from 128 to 3072
            config=genai.types.EmbedContentConfig(output_dimensionality=3072)
        )
        
        # Extract embeddings from batch response
        for embedding in result.embeddings:
            embeddings_list.append(embedding.values)
        
        print(f"  ✓ Successfully processed {len(batch_texts)} embeddings")
        
    except Exception as e:
        print(f"  ✗ Error generating batch embeddings: {e}")
        # Add None for failed embeddings
        embeddings_list.extend([None] * len(batch_texts))
    
    # Rate limiting: wait before next batch (except for the last batch)
    if batch_num < total_batches - 1:
        print(f"  Waiting {RATE_LIMIT_DELAY} seconds before next batch...")
        time.sleep(RATE_LIMIT_DELAY)

# Display sample results (only first 3 for brevity)
print("\n" + "="*60)
print("Sample Results:")
print("="*60)
for idx in range(min(3, len(job_postings_batch))):
    job_data = job_postings_batch.iloc[idx]
    if idx < len(embeddings_list) and embeddings_list[idx] is not None:
        print(f"Job {idx + 1}: '{job_data['title']}'")
        print(f"  Embedding length: {len(embeddings_list[idx])}")
        print(f"  First 10 values: {embeddings_list[idx][:10]}")
        print()

# Add embeddings to DataFrame
job_postings_batch['embedding'] = embeddings_list

# Display DataFrame info
print("\nDataFrame shape:", job_postings_batch.shape)
print("Columns:", job_postings_batch.columns.tolist())
print(f"\nNon-null embeddings: {sum(1 for e in embeddings_list if e is not None)}/{len(embeddings_list)}")


In [ ]:
# remove duplicates
job_postings_batch = job_postings_batch.drop_duplicates(subset=['title', 'description', 'location']).reset_index(drop=True)

# export to pickle
import pickle

job_postings_batch = job_postings_batch.reset_index(drop=True)
# Save embeddings to pickle file
output_file = 'assets/models/ds_v1/embeddings_title_summary_1000_3072d_v2.pkl'
output_file = output_file
# check if file exists, if yes, load existing data and append new data, then save
if os.path.exists(output_file):
    existing_df = pd.read_pickle(output_file)
    combined_df = pd.concat([existing_df, job_postings_batch], ignore_index=True)
    combined_df.to_pickle(output_file)
    print(f"✓ Appended {len(job_postings_batch)} job postings to existing file '{output_file}' (total {len(combined_df)})")
else:
    job_postings_batch.to_pickle(output_file)
    print(f"✓ Saved {len(job_postings_batch)} job postings with embeddings to '{output_file}'")

# Verify the saved file
loaded_df = pd.read_pickle(output_file)
print(f"\nVerification:")
print(f"  Loaded shape: {loaded_df.shape}")
print(f"  Non-null embeddings: {loaded_df['embedding'].notna().sum()}")
print(f"  Sample embedding dimensions: {len(loaded_df['embedding'].iloc[0]) if loaded_df['embedding'].iloc[0] is not None else 'N/A'}")


# EDA / Test sets

### Ground-truth entry-level jobs (exclude interns)

### RegEx Method

In [ ]:
from ground_truth_generation import RegexGroundTruthGenerator

regex_gen = RegexGroundTruthGenerator()
entry_level = regex_gen.filter_entry_level(first_1000)
entry_level

In [ ]:
data_business_analytics = regex_gen.filter_data_business_analytics(first_1000)
finance_analytics = regex_gen.filter_finance_analytics(first_1000)
marketing_analytics = regex_gen.filter_marketing_analytics(first_1000)

print(f"Data + Business Analytics: {len(data_business_analytics)} jobs")
print(f"Finance Analytics: {len(finance_analytics)} jobs")
print(f"Marketing Analytics: {len(marketing_analytics)} jobs")

finance_analytics


### LLM-Labeled Ground Truth

In [2]:
data_analytics = pd.read_csv('assets/ground_truths/ds_v2/llm_generated/data_analytics.csv')
entry_level = pd.read_csv('assets/ground_truths/ds_v2/llm_generated/entry_level.csv')
finance_accounting = pd.read_csv('assets/ground_truths/ds_v2/llm_generated/finance_accounting.csv')
internship = pd.read_csv('assets/ground_truths/ds_v2/llm_generated/internship.csv')
marketing = pd.read_csv('assets/ground_truths/ds_v2/llm_generated/marketing.csv')
software_engineering = pd.read_csv('assets/ground_truths/ds_v2/llm_generated/software_engineering.csv')
strategy_product_consulting = pd.read_csv('assets/ground_truths/ds_v2/llm_generated/strategy_product_consulting.csv')
all_ground_truths = pd.read_csv('assets/ground_truths/ds_v2/llm_generated/checkpoint_labels.csv')

print('data_analytics: ', len(data_analytics))
print('entry_level: ', len(entry_level))
print('finance_accounting: ', len(finance_accounting))
print('internship: ', len(internship))
print('marketing: ', len(marketing))
print('software_engineering: ', len(software_engineering))
print('strategy_product_consulting: ', len(strategy_product_consulting))


data_analytics:  116
entry_level:  70
finance_accounting:  209
internship:  64
marketing:  281
software_engineering:  296
strategy_product_consulting:  155


# Indexing

# Evaluation (F1/Recall/Precision)

### Single Model Evaluation

In [4]:
from sklearn.metrics import ndcg_score as sk_ndcg_score


def get_retrieved_ids_by_k(query_text, faiss_idx, category, max_k=100):
    """
    Perform FAISS search once and return retrieved IDs and similarity scores.
    
    Args:
        query_text (str): The search query text
        faiss_idx (FAISSIndex): The FAISS index instance to search
        max_k (int): Maximum K value to retrieve
    
    Returns:
        tuple: (list of retrieved job_posting_ids, list of similarity_scores)
    """
    search_results = faiss_idx.search_qwen(None, category, query_text, top_k=max_k)
    retrieved_ids = search_results['job_posting_id'].tolist()
    similarity_scores = search_results['similarity_score'].tolist()
    return retrieved_ids, similarity_scores


def calculate_recall_at_k(retrieved_ids, ground_truth_ids, k_values=[5, 10, 20, 30, 50]):
    """
    Calculate Recall@K for given retrieved IDs against ground truth job IDs.
    """
    recall_scores = {}
    for k in k_values:
        top_k_ids = retrieved_ids[:k]
        relevant_retrieved = len(set(top_k_ids) & set(ground_truth_ids))
        recall_at_k = relevant_retrieved / len(ground_truth_ids) if len(ground_truth_ids) > 0 else 0
        recall_scores[f'Recall@{k}'] = recall_at_k
    return recall_scores


def calculate_precision_at_k(retrieved_ids, ground_truth_ids, k_values=[5, 10, 20, 30, 50]):
    """
    Calculate Precision@K for given retrieved IDs against ground truth job IDs.
    """
    precision_scores = {}
    for k in k_values:
        top_k_ids = retrieved_ids[:k]
        relevant_retrieved = len(set(top_k_ids) & set(ground_truth_ids))
        precision_at_k = relevant_retrieved / k if k > 0 else 0
        precision_scores[f'Precision@{k}'] = precision_at_k
    return precision_scores


def calculate_ndcg_at_k(retrieved_ids, ground_truth_ids, similarity_scores, k_values=(5, 10, 20, 30, 50)):
    """
    Calculate NDCG@K for given retrieved IDs against ground truth job IDs.

    Uses the full retrieved list (not sliced to :k) with the k= parameter so that
    relevant items ranked outside top-k are penalised — unlike slicing where
    NDCG@k can be 1.0 even with an irrelevant item in the window.

    Args:
        retrieved_ids (list): Ordered list of retrieved job_posting_ids (descending similarity).
        ground_truth_ids (list): List of relevant job_posting_ids.
        similarity_scores (list): Similarity score for each retrieved_id (same order).
        k_values (tuple): K values to evaluate.

    Returns:
        dict: {f'NDCG@{k}': float, ...}
    """
    ndcg_scores = {}
    y_true_full = np.isin(retrieved_ids, ground_truth_ids).astype(int)
    y_score_full = np.array(similarity_scores)
    for k in k_values:
        if y_true_full.sum() == 0:
            ndcg_scores[f'NDCG@{k}'] = 0.0
        else:
            ndcg_scores[f'NDCG@{k}'] = float(sk_ndcg_score([y_true_full], [y_score_full], k=k))
    return ndcg_scores


# --- Configuration ---
category = 'strategy_product_consulting'
with open('assets/category_map.yaml', 'r') as f:
    _category_map = yaml.safe_load(f)
query_text = _category_map['categories'][category]['query']


k_values = [10]
max_k = max(k_values)

# --- Model input: load embedding pkl and initialize FAISS ---
df_model = pd.read_pickle('assets/models/ds_v2/qwen_1024d_0.6B.pkl')

faiss_eval = FAISSIndex(df_model)
faiss_eval.build_index()
print(f"✓ FAISS index built with {len(df_model)} job postings (dim={faiss_eval.dimension})")

# --- Perform search ---
retrieved_ids, similarity_scores = get_retrieved_ids_by_k(query_text, faiss_eval, category, max_k=max_k)

# --- Ground truth ---
ground_truth = strategy_product_consulting.copy()

ground_truth_ids = ground_truth['job_posting_id'].tolist()
print(f"Ground truth: {len(ground_truth_ids)} entry-level job postings")
print(f"Query: '{query_text}'\n")

# --- Calculate metrics ---
recall_results = calculate_recall_at_k(
    retrieved_ids=retrieved_ids,
    ground_truth_ids=ground_truth_ids,
    k_values=k_values
)

precision_results = calculate_precision_at_k(
    retrieved_ids=retrieved_ids,
    ground_truth_ids=ground_truth_ids,
    k_values=k_values
)

ndcg_results = calculate_ndcg_at_k(
    retrieved_ids=retrieved_ids,
    ground_truth_ids=ground_truth_ids,
    similarity_scores=similarity_scores,
    k_values=k_values
)

# --- Display metrics ---
print("=" * 50)
print("Results@K:")
print("=" * 50)
for metric, score in recall_results.items():
    print(f"{metric}: {score:.4f} ({score*100:.2f}%)")
for metric, score in precision_results.items():
    print(f"{metric}: {score:.4f} ({score*100:.2f}%)")
for metric, score in ndcg_results.items():
    print(f"{metric}: {score:.4f} ({score*100:.2f}%)")

# --- Retrieve results with is_entry_level flag ---
search_results = faiss_eval.search_qwen(None, category, query_text, top_k=max_k)
ground_truth_id_set = set(ground_truth_ids)
search_results['correct_output'] = search_results['job_posting_id'].isin(ground_truth_id_set)

print(f"\n✓ Retrieved {len(search_results)} jobs (top {max_k})")
print(f"  True matches: {search_results['correct_output'].sum()}/{len(search_results)}")
print("\nRetrieved jobs:")
# display all rows of search_results
pd.set_option('display.max_rows', None)

display(search_results[['job_posting_id', 'title', 'similarity_score', 'correct_output', 'description']])

✓ FAISS index built with 1000 job postings (dim=1024)
Ground truth: 155 entry-level job postings
Query: 'strategy consultant product manager business strategy management consulting product strategy product operations go-to-market'

Results@K:
Recall@10: 0.0387 (3.87%)
Precision@10: 0.6000 (60.00%)
NDCG@10: 0.8976 (89.76%)

✓ Retrieved 10 jobs (top 10)
  True matches: 6/10

Retrieved jobs:


,job_posting_id,title,similarity_score,correct_output,description
0,fea4f27e-03f2-440f-97cb-743559f38502,Product Manager AI,0.756990,True,Role Overview Manage roadmaps across all Platf...
1,dc34526d-0ada-4472-93f0-5cdec113eed4,PR Specialist,0.581647,False,Job description PR Specialist – koderAI Applic...
2,3a35f067-f71c-41ba-bb6b-a464c9b18473,Senior Strategy Manager,0.580874,True,"About Us: Advanced Systems Group, LLC enables ..."
3,2efa6de5-dc99-42c0-b79c-bd651b51de1a,Senior Strategist,0.573526,True,THE ROLE We are looking for a Senior Strategis...
4,69c0e192-6747-440d-8634-2ba4038c01de,Product Owner - AI Products & Strategy,0.568090,True,"Hello, Good Morning! Hope you are doing well. ..."
5,8bd7024b-b51e-4f16-94be-e0dac42a16f5,Marketing Coordinator,0.565461,False,Looking for the Brightest! At Structure Tone S...
6,1a52e57e-2475-40ec-86d2-c8b8de8ff750,Senior Director Analyst AI and D&A,0.562307,True,"Senior Director, Analyst – Data & Analytics an..."
7,70e3e5c7-bf88-4cb4-aba9-5a7595bf6569,AI Product Owner,0.560580,True,"New York City, New York 07008 Posted February ..."
8,22efb312-83c0-4c92-a033-03a6fd051d66,"Senior Analyst, Search Engine Optimization",0.560207,False,Company Description Digitas is the Networked E...
9,5581695c-a148-49b8-90a3-08f3634ad97d,"Senior Manager, Category Development",0.557728,False,Staples is business to business. You’re what b...


### Model comparison

In [7]:
import pandas as pd
import numpy as np
import re
import os
from faiss_index import FAISSIndex
from sklearn.metrics import ndcg_score as sk_ndcg_score


def model_name_from_path(pkl_path: str) -> str:
    """Derive a short model label from a pickle filename.

    e.g. 'assets/models/ds_v1/embeddings_title_summary_1000_768d_v2.pkl' -> 'title_summary_768d'
    """
    name = os.path.basename(pkl_path).replace('.pkl', '')  # embeddings_title_summary_1000_768d_v2
    name = re.sub(r'^embeddings_', '', name)               # title_summary_1000_768d_v2
    name = re.sub(r'_\d+(?=_)', '', name)                  # title_summary_768d_v2
    name = re.sub(r'_v\d+$', '', name)                     # title_summary_768d
    return name


# Add or remove paths here to control which models are compared
pkl_paths = [
    # 'assets/models/ds_v1/embeddings_title_1000_768d.pkl',
    # 'assets/models/ds_v1/embeddings_title_description_1500_768d.pkl',
    # 'assets/models/ds_v1/embeddings_title_summary_1000_768d_v2.pkl',
    # 'assets/models/ds_v1/embeddings_title_summary_1000_3072d_v2.pkl',
    'assets/models/ds_v2/qwen_1024d_0.6B.pkl',
    'assets/models/ds_v2/qwen_4096d_8B.pkl',
]

models = [(path, pd.read_pickle(path)[:1000]) for path in pkl_paths]

with open('assets/category_map.yaml', 'r') as f:
    _category_map = yaml.safe_load(f)

_category_vars = {
    'entry_level': entry_level,
    'internship': internship,
    'data_analytics': data_analytics,
    'software_engineering': software_engineering,
    'finance_accounting': finance_accounting,
    'marketing': marketing,
    'strategy_product_consulting': strategy_product_consulting,
}

# Define ground truths and their corresponding query texts
ground_truths = {
    name: {
        'ids': df['job_posting_id'].tolist(),
        'query': _category_map['categories'][name]['query'],
    }
    for name, df in _category_vars.items()
}

k_values = [10, 50, 100]

# Function to evaluate a single model
def evaluate_model(df_embeddings, model_name, category, query_text, ground_truth_ids, k_values=k_values):
    """Evaluate recall and precision for a given embedding model"""
    
    # Initialize FAISS index for this model
    faiss_idx = FAISSIndex(df_embeddings)
    faiss_idx.build_index()
    
    # Get max K for search
    max_k = max(k_values)
    
    # Search
    search_results = faiss_idx.search_qwen(None, category, query_text, top_k=max_k)
    retrieved_ids = search_results['job_posting_id'].tolist()
    similarity_scores = search_results['similarity_score'].tolist()
    
    results = {'Model': model_name}

    # Pre-compute full relevance and score arrays for NDCG (uses full list + k= param)
    y_true_full = np.isin(retrieved_ids, ground_truth_ids).astype(int)
    y_score_full = np.array(similarity_scores)
    
    # Calculate metrics for each K
    for k in k_values:
        top_k_ids = retrieved_ids[:k]
        relevant_retrieved = len(set(top_k_ids) & set(ground_truth_ids))
        
        # Recall@K
        recall = relevant_retrieved / len(ground_truth_ids) if len(ground_truth_ids) > 0 else 0
        results[f'Recall@{k}'] = recall
        
        # Precision@K
        precision = relevant_retrieved / k if k > 0 else 0
        results[f'Precision@{k}'] = precision

        # F1@K
        f1 = (2 * precision * recall) / (precision + recall) if (precision + recall) > 0 else 0
        results[f'F1@{k}'] = f1

        # NDCG@K
        if y_true_full.sum() == 0:
            results[f'NDCG@{k}'] = 0.0
        else:
            results[f'NDCG@{k}'] = float(sk_ndcg_score([y_true_full], [y_score_full], k=k))
    
    return results

# Evaluate all ground truths
for category_name, category_data in ground_truths.items():
    ground_truth_ids = category_data['ids']
    query_text = category_data['query']

    all_results = [
        evaluate_model(df, model_name_from_path(path), category_name, query_text, ground_truth_ids)
        for path, df in models
    ]

    # Create comparison DataFrame
    comparison_df = pd.DataFrame(all_results)
    
    # Reorder columns: Model, then per-k metrics grouped together
    ordered_cols = ['Model']
    for k in k_values:
        # ordered_cols.extend([f'Recall@{k}', f'Precision@{k}'])
        ordered_cols.extend([f'Recall@{k}', f'Precision@{k}', f'NDCG@{k}'])
    
    comparison_df = comparison_df[ordered_cols]
    
    # Display category info and results together
    print(f"\n{'='*100}")
    print(f"Category: {category_name}")
    print(f"{'='*100}")
    print(f"Ground truth: {len(ground_truth_ids)} job postings")
    print(f"Query: '{query_text}'\n")
    
    print("="*100)
    print(f"Table: F1@K Comparison - {category_name}")
    print("="*100)
    print(comparison_df.to_string(index=False))
    print()



Category: entry_level
Ground truth: 70 job postings
Query: 'junior entry level new grad associate early career trainee recent graduate engineer analyst developer designer consultant coordinator specialist'

Table: F1@K Comparison - entry_level
          Model  Recall@10  Precision@10  NDCG@10  Recall@50  Precision@50  NDCG@50  Recall@100  Precision@100  NDCG@100
qwen_1024d_0.6B   0.114286           0.8 0.841493   0.314286          0.44 0.842831    0.371429           0.26  0.920713
  qwen_4096d_8B   0.114286           0.8 0.820523   0.214286          0.30 0.677877    0.342857           0.24  0.861894


Category: internship
Ground truth: 64 job postings
Query: 'intern internship co-op summer analyst trainee undergraduate student entry internship program'

Table: F1@K Comparison - internship
          Model  Recall@10  Precision@10  NDCG@10  Recall@50  Precision@50  NDCG@50  Recall@100  Precision@100  NDCG@100
qwen_1024d_0.6B   0.140625           0.9 0.926636   0.609375          0.78 0.8